# Eigenvalues & Eigenvectors

特征值与特征向量。从定义到特征分解，从对称矩阵正交对角化到 PCA 入门，全程配 2D 可视化建立几何直觉。

> 本节课代码为主，每个数学概念都配多个可运行实例。建议逐段运行，重点观察可视化结果。

## 0. 环境配置与导入

确认 PyTorch 可用，设置随机种子保证可复现。

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 特征值与特征向量的定义

### 1.1 数学定义

对于方阵 $A \in \mathbb{R}^{n \times n}$，如果存在非零向量 $v \in \mathbb{R}^n$ 和标量 $\lambda \in \mathbb{C}$，使得：

$$
Av = \lambda v
$$

则称：
- $\lambda$ 为 $A$ 的**特征值**（eigenvalue）
- $v$ 为对应于 $\lambda$ 的**特征向量**（eigenvector）

**几何直觉**：矩阵 $A$ 代表一个线性变换。大多数向量经过变换后方向会改变，但**特征向量经过变换后方向不变（或恰好反向）**，只在长度上缩放 $\lambda$ 倍。

- $|\lambda| > 1$：特征向量方向被拉伸
- $|\lambda| < 1$：特征向量方向被压缩
- $\lambda > 0$：方向不变
- $\lambda < 0$：方向反转
- $\lambda = 0$：该方向被压缩到零（矩阵不可逆）

In [ ]:
# 一个简单的例子：对角矩阵
A = torch.tensor([[3.0, 0.0],
                  [0.0, 1.0]])

# 标准基向量 e1 = [1,0], e2 = [0,1]
e1 = torch.tensor([1.0, 0.0])
e2 = torch.tensor([0.0, 1.0])

print("A =\n", A)
print("\nA @ e1 =", A @ e1, "= 3 * e1 → λ=3, e1 是特征向量")
print("A @ e2 =", A @ e2, "= 1 * e2 → λ=1, e2 是特征向量")

# 非特征向量：方向会改变
v = torch.tensor([1.0, 1.0])
print("\n非特征向量 v =", v)
print("A @ v =", A @ v, "→ 方向改变了，不是特征向量")
# [1,1] → [3,1]，方向从 45° 变成 arctan(1/3) ≈ 18.4°

### 1.2 2D 可视化：特征向量是不变方向

用网格可视化线性变换，直观感受特征向量的"方向不变"特性。

In [ ]:
%matplotlib inline

def visualize_eigenvectors(A, title, ax=None):
    """可视化 2D 线性变换：网格 + 特征向量方向"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    # 生成网格
    grid_range = torch.linspace(-2, 2, 9)
    for i in grid_range:
        # 水平线
        h_line = torch.stack([torch.linspace(-2, 2, 50), torch.full((50,), i)], dim=1)
        h_transformed = (A @ h_line.T).T
        ax.plot(h_line[:, 0], h_line[:, 1], 'b-', alpha=0.2)
        ax.plot(h_transformed[:, 0], h_transformed[:, 1], 'r-', alpha=0.3)
        # 垂直线
        v_line = torch.stack([torch.full((50,), i), torch.linspace(-2, 2, 50)], dim=1)
        v_transformed = (A @ v_line.T).T
        ax.plot(v_line[:, 0], v_line[:, 1], 'b-', alpha=0.2)
        ax.plot(v_transformed[:, 0], v_transformed[:, 1], 'r-', alpha=0.3)
    
    # 计算特征向量并画出
    eigenvalues, eigenvectors = torch.linalg.eig(A)
    for i in range(2):
        lam = eigenvalues[i].real.item()
        vec = eigenvectors[:, i].real
        if abs(eigenvalues[i].imag.item()) > 1e-6:
            continue  # 跳过复特征值
        # 原始特征向量
        ax.arrow(0, 0, vec[0].item(), vec[1].item(),
                 head_width=0.15, color='green', linewidth=2, label=f'v{i+1} (λ={lam:.2f})')
        # 变换后的特征向量（应该共线）
        transformed = A @ vec
        ax.arrow(0, 0, transformed[0].item(), transformed[1].item(),
                 head_width=0.15, color='purple', linewidth=2, linestyle='--',
                 label=f'A@v{i+1} = {lam:.2f}v{i+1}')
    
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    return ax

# 对角矩阵：特征向量就是坐标轴方向
A1 = torch.tensor([[3.0, 0.0], [0.0, 0.5]])
fig, ax = plt.subplots(figsize=(6, 6))
visualize_eigenvectors(A1, "对角矩阵: λ=[3, 0.5], 特征向量=坐标轴", ax)
plt.tight_layout()
plt.show()

print("绿色箭头 = 原始特征向量，紫色虚线 = 变换后的特征向量")
print("两者共线 → 特征向量方向不变，只缩放长度")

In [ ]:
# 非对角对称矩阵：特征向量不是坐标轴方向
A2 = torch.tensor([[2.0, 1.0],
                    [1.0, 2.0]])

fig, ax = plt.subplots(figsize=(6, 6))
visualize_eigenvectors(A2, "对称矩阵 [[2,1],[1,2]]: 特征向量在对角线方向", ax)
plt.tight_layout()
plt.show()

# 手算验证：[[2,1],[1,2]] 的特征值
# det(A-λI) = (2-λ)^2 - 1 = λ²-4λ+3 = (λ-1)(λ-3) = 0
# λ=1 时: (A-I)v=0 → [[1,1],[1,1]]v=0 → v=[1,-1]
# λ=3 时: (A-3I)v=0 → [[-1,1],[1,-1]]v=0 → v=[1,1]
print("预期特征值: λ=1 (v=[1,-1]), λ=3 (v=[1,1])")
eigvals, eigvecs = torch.linalg.eig(A2)
print("PyTorch 特征值:", eigvals.real)
print("PyTorch 特征向量（列）:\n", eigvecs.real)

## 2. 特征方程与特征多项式

### 2.1 特征方程

从定义 $Av = \lambda v$ 出发：

$$
Av - \lambda v = 0 \implies (A - \lambda I)v = 0
$$

要使这个齐次线性方程组有**非零解** $v$，系数矩阵必须是奇异的：

$$
\det(A - \lambda I) = 0
$$

这就是**特征方程**（characteristic equation）。

### 2.2 特征多项式

$p(\lambda) = \det(A - \lambda I)$ 是关于 $\lambda$ 的 $n$ 次多项式，称为**特征多项式**（characteristic polynomial）。

对于 2×2 矩阵 $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$：

$$
p(\lambda) = \lambda^2 - (a+d)\lambda + (ad-bc) = \lambda^2 - \text{tr}(A)\lambda + \det(A)
$$

特征值就是特征多项式的根。

In [ ]:
# 2x2 矩阵的特征多项式手算 + 代码验证
A = torch.tensor([[2.0, 1.0],
                  [1.0, 2.0]])

a, b = A[0, 0].item(), A[0, 1].item()
c, d = A[1, 0].item(), A[1, 1].item()

tr_A = a + d
det_A = a * d - b * c

print(f"A = [[{a}, {b}], [{c}, {d}]]")
print(f"tr(A) = {tr_A}")
print(f"det(A) = {det_A}")
print(f"特征多项式: p(λ) = λ² - {tr_A}λ + {det_A}")
print(f"即: λ² - {tr_A}λ + {det_A} = 0")

# 用求根公式解
discriminant = tr_A**2 - 4 * det_A
lambda1 = (tr_A + discriminant**0.5) / 2
lambda2 = (tr_A - discriminant**0.5) / 2
print(f"\n求根公式: λ = ({tr_A} ± √{discriminant}) / 2")
print(f"λ1 = {lambda1}, λ2 = {lambda2}")

# 与 PyTorch 结果对比
eigvals, _ = torch.linalg.eig(A)
print(f"\nPyTorch 特征值: {eigvals.real.tolist()}")
print(f"匹配: λ1≈{eigvals[0].real.item():.4f}, λ2≈{eigvals[1].real.item():.4f}")

In [ ]:
# 验证特征方程：det(A - λI) ≈ 0
A = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
eigvals, _ = torch.linalg.eig(A)

I = torch.eye(2)
for i in range(2):
    lam = eigvals[i].real
    det_val = torch.det(A - lam * I)
    print(f"λ{i+1} = {lam.item():.4f}, det(A - λ{i+1}I) = {det_val.item():.6f} (≈0)")

# 3x3 矩阵的特征多项式（次数为 3）
B = torch.tensor([[4.0, -2.0, 1.0],
                  [-2.0, 3.0, -1.0],
                  [1.0, -1.0, 2.0]])
eigvals_B, _ = torch.linalg.eig(B)
print("\n3x3 对称矩阵的特征值:", sorted(eigvals_B.real.tolist()))
print("特征值之和 = tr(B) =", eigvals_B.real.sum().item(), "vs", torch.trace(B).item())
print("特征值之积 = det(B) =", eigvals_B.real.prod().item(), "vs", torch.det(B).item())

## 3. PyTorch 求解特征值与特征向量

### 3.1 torch.linalg.eig

`torch.linalg.eig(A)` 返回：
- `eigenvalues`：形状 `[n]`，特征值（可能是复数，dtype=complex）
- `eigenvectors`：形状 `[n, n]`，第 $i$ 列是对应第 $i$ 个特征值的特征向量

**注意**：
- 特征向量已按 L2 范数归一化（$\|v_i\|_2 = 1$）
- 特征值和特征向量的顺序不保证排序
- 特征向量的符号不保证（$v$ 和 $-v$ 都是特征向量）
- 非对称矩阵可能有复数特征值

In [ ]:
A = torch.tensor([[4.0, -2.0],
                  [1.0, 1.0]])

eigenvalues, eigenvectors = torch.linalg.eig(A)

print("A =\n", A)
print("\neigenvalues:", eigenvalues)
print("eigenvalues dtype:", eigenvalues.dtype)  # complex64/complex128
print("\neigenvectors (每列是一个特征向量):\n", eigenvectors)
print("eigenvectors shape:", eigenvectors.shape)

# 取实部（这个例子特征值是实数）
print("\n特征值（实部）:", eigenvalues.real.tolist())
print("特征向量（实部）:\n", eigenvectors.real)

# 验证特征向量已归一化
for i in range(2):
    vec = eigenvectors[:, i]
    norm = torch.linalg.norm(vec)
    print(f"||v{i+1}|| = {norm.item():.6f} (≈1)")

### 3.2 验证 Av = λv

对每个特征值-特征向量对，验证定义式 $Av = \lambda v$。

In [ ]:
A = torch.tensor([[4.0, -2.0],
                  [1.0, 1.0]])
eigenvalues, eigenvectors = torch.linalg.eig(A)
A_complex = A.to(torch.complex64)  # 特征向量是复数，A 需转成 complex

for i in range(2):
    lam = eigenvalues[i]
    v = eigenvectors[:, i]
    
    Av = A_complex @ v
    lambda_v = lam * v
    
    print(f"--- 特征对 {i+1} ---")
    print(f"λ = {lam.item()}")
    print(f"v = {v.tolist()}")
    print(f"A @ v = {Av.tolist()}")
    print(f"λ * v = {lambda_v.tolist()}")
    print(f"匹配? {torch.allclose(Av, lambda_v, atol=1e-5)}")
    print()

In [ ]:
# 3x3 矩阵的完整验证
B = torch.tensor([[2.0, 1.0, 0.0],
                  [1.0, 2.0, 1.0],
                  [0.0, 1.0, 2.0]])  # 三对角对称矩阵

eigenvalues_B, eigenvectors_B = torch.linalg.eig(B)

print("B =\n", B)
print("\n特征值:", eigenvalues_B.real.tolist())
print("\n逐对验证 Av = λv:")
for i in range(3):
    lam = eigenvalues_B[i].real
    v = eigenvectors_B[:, i].real
    Av = B @ v
    lambda_v = lam * v
    match = torch.allclose(Av, lambda_v, atol=1e-5)
    print(f"  λ{i+1}={lam:.4f}, ||A@v - λv||={torch.norm(Av-lambda_v).item():.2e}, 匹配={match}")

### 3.3 只求特征值：torch.linalg.eigvals

如果只需要特征值，不需要特征向量，用 `torch.linalg.eigvals(A)` 更高效。

In [ ]:
A = torch.randn(5, 5)  # 随机 5x5 矩阵

# 只算特征值
eigvals_only = torch.linalg.eigvals(A)
print("eigvals 形状:", eigvals_only.shape)
print("特征值（实部）:", eigvals_only.real.tolist())

# 与 eig 对比
eigvals_full, _ = torch.linalg.eig(A)
print("\neig 返回的特征值（实部）:", eigvals_full.real.tolist())
print("两者一致?", torch.allclose(eigvals_only.real.sort().values,
                                  eigvals_full.real.sort().values, atol=1e-5))

### 3.4 复数特征值

非对称矩阵（特别是旋转矩阵）可能有**复数特征值**。此时特征向量也是复数。

复数特征值总是以共轭对 $a \pm bi$ 的形式出现（对于实矩阵）。

In [ ]:
# 旋转矩阵：特征值一定是复数（除非旋转 0° 或 180°）
import math
theta = math.pi / 3  # 60°
R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta),  math.cos(theta)]])

eigenvalues, eigenvectors = torch.linalg.eig(R)

print("旋转矩阵 R (60°) =\n", R)
print("\n特征值:", eigenvalues)
print(f"λ1 = {eigenvalues[0].item():.4f} = cos60° + i·sin60° = 0.5 + 0.866i")
print(f"λ2 = {eigenvalues[1].item():.4f} = cos60° - i·sin60° = 0.5 - 0.866i")
print("\n复数特征值的模:", eigenvalues.abs().tolist(), "(=1, 旋转不改变长度)")

# 验证：复数特征值的模 = 1（正交矩阵的特征值模为 1）
print("\n|λ1| =", eigenvalues[0].abs().item())
print("|λ2| =", eigenvalues[1].abs().item())

# 特征向量也是复数
print("\n特征向量:\n", eigenvectors)
print("特征向量 dtype:", eigenvectors.dtype)

In [ ]:
# 另一个有复特征值的非对称矩阵
A = torch.tensor([[1.0, -2.0],
                  [3.0, 1.0]])

eigenvalues, eigenvectors = torch.linalg.eig(A)
print("A =\n", A)
print("\n特征值:", eigenvalues)
print("实部:", eigenvalues.real.tolist())
print("虚部:", eigenvalues.imag.tolist())

# 验证共轭对：λ2 = conjugate(λ1)
print("\nλ2 == conj(λ1)?", torch.allclose(eigenvalues[1], eigenvalues[0].conj()))

# 验证 Av = λv（复数版本）
for i in range(2):
    lam = eigenvalues[i]
    v = eigenvectors[:, i]
    Av = A.to(torch.complex64) @ v
    print(f"λ{i+1}={lam.item()}, A@v≈λv? {torch.allclose(Av, lam*v, atol=1e-5)}")

### 3.5 特征向量的符号不确定性

特征向量 $v$ 和 $-v$ 都满足 $Av = \lambda v$，因此 PyTorch 返回的特征向量符号是不确定的。不同运行、不同库可能返回相反符号，但都是正确的。

In [ ]:
A = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
eigenvalues, eigenvectors = torch.linalg.eig(A)

v = eigenvectors[:, 0].real
print("v =", v.tolist())
print("-v =", (-v).tolist())
print("v 和 -v 都是特征向量:")
print(f"  A @ v = {(A @ v).tolist()}")
print(f"  A @ (-v) = {(A @ -v).tolist()}")
print(f"  λ * v = {(eigenvalues[0].real * v).tolist()}")
print(f"  λ * (-v) = {(eigenvalues[0].real * -v).tolist()}")
print("两者都满足 Av = λv，符号不影响正确性")

## 4. 特征值的基本性质

### 4.1 迹 = 特征值之和

$$
\text{tr}(A) = \sum_{i=1}^{n} \lambda_i
$$

矩阵的迹等于所有特征值之和（包括复特征值的实部之和）。

In [ ]:
# 验证 tr(A) = Σ λ_i
A = torch.tensor([[4.0, -2.0, 1.0],
                  [1.0, 2.0, -1.0],
                  [0.0, 1.0, 3.0]])

eigenvalues, _ = torch.linalg.eig(A)
print("A =\n", A)
print("\n特征值:", eigenvalues)
print("特征值之和:", eigenvalues.sum().item())
print("tr(A):", torch.trace(A).item())
print("匹配?", torch.allclose(eigenvalues.sum().real, torch.trace(A), atol=1e-5))

# 随机矩阵多次验证
print("\n--- 随机 4x4 矩阵验证 ---")
for trial in range(3):
    M = torch.randn(4, 4)
    evals, _ = torch.linalg.eig(M)
    sum_evals = evals.sum().real.item()
    trace_M = torch.trace(M).item()
    print(f"  试验{trial+1}: Σλ={sum_evals:.6f}, tr={trace_M:.6f}, 差={abs(sum_evals-trace_M):.2e}")

### 4.2 行列式 = 特征值之积

$$
\det(A) = \prod_{i=1}^{n} \lambda_i
$$

矩阵的行列式等于所有特征值之积。

In [ ]:
# 验证 det(A) = Π λ_i
A = torch.tensor([[2.0, 1.0, 0.0],
                  [1.0, 2.0, 1.0],
                  [0.0, 1.0, 2.0]])

eigenvalues, _ = torch.linalg.eig(A)
prod_evals = eigenvalues.prod().real.item()
det_A = torch.det(A).item()

print("A =\n", A)
print("\n特征值:", eigenvalues.real.tolist())
print("特征值之积:", prod_evals)
print("det(A):", det_A)
print("匹配?", torch.allclose(eigenvalues.prod().real, torch.det(A), atol=1e-5))

# 奇异矩阵：至少一个特征值为 0
S = torch.tensor([[1.0, 2.0, 3.0],
                  [2.0, 4.0, 6.0],
                  [1.0, 1.0, 1.0]])  # 第2行 = 2*第1行，奇异
eigvals_S, _ = torch.linalg.eig(S)
print("\n奇异矩阵 S 的特征值:", eigvals_S.real.tolist())
print("det(S) =", torch.det(S).item(), "(≈0)")
print("特征值之积 =", eigvals_S.prod().real.item(), "(≈0)")
print("→ 奇异矩阵 ⟺ 至少一个特征值为 0")

### 4.3 可逆性与特征值

- $A$ 可逆 $\iff$ 所有特征值 $\lambda_i \neq 0$
- $A$ 奇异（不可逆）$\iff$ 至少一个特征值 $= 0$
- $A^{-1}$ 的特征值是 $1/\lambda_i$（对应相同的特征向量）

In [ ]:
A = torch.tensor([[4.0, 1.0],
                  [2.0, 3.0]])
eigenvalues_A, eigenvectors_A = torch.linalg.eig(A)

print("A 的特征值:", eigenvalues_A.real.tolist())
print("A 可逆? det(A) =", torch.det(A).item(), "≠ 0 → 可逆")

# A^{-1} 的特征值 = 1/λ
A_inv = torch.linalg.inv(A)
eigenvalues_inv, _ = torch.linalg.eig(A_inv)

print("\nA^{-1} 的特征值:", eigenvalues_inv.real.tolist())
print("1/λ(A):", (1.0 / eigenvalues_A.real).tolist())
print("匹配?", torch.allclose(eigenvalues_inv.real.sort().values,
                              (1.0/eigenvalues_A.real).sort().values, atol=1e-5))

# 验证：A^{-1} 的特征向量与 A 相同
v = eigenvectors_A[:, 0].real
lam = eigenvalues_A[0].real
print(f"\n验证 A^{{-1}} @ v = (1/λ) @ v:")
print(f"  A^{{-1}} @ v = {(A_inv @ v).tolist()}")
print(f"  (1/{lam:.4f}) @ v = {(1/lam * v).tolist()}")

### 4.4 三角矩阵的特征值 = 对角线元素

上三角或下三角矩阵的特征值就是其主对角线元素。

In [ ]:
# 上三角矩阵
U = torch.tensor([[3.0, 2.0, 1.0],
                  [0.0, 5.0, 4.0],
                  [0.0, 0.0, 2.0]])

eigenvalues_U, _ = torch.linalg.eig(U)
print("上三角矩阵 U =\n", U)
print("对角线元素:", [U[i, i].item() for i in range(3)])
print("特征值:", sorted(eigenvalues_U.real.tolist()))
print("匹配?", torch.allclose(eigenvalues_U.real.sort().values,
                              U.diagonal().sort().values, atol=1e-5))

# 下三角矩阵
L = torch.tensor([[2.0, 0.0, 0.0],
                  [1.0, 4.0, 0.0],
                  [3.0, 2.0, 1.0]])
eigenvalues_L, _ = torch.linalg.eig(L)
print("\n下三角矩阵 L 的对角线:", [L[i, i].item() for i in range(3)])
print("特征值:", sorted(eigenvalues_L.real.tolist()))
print("匹配?", torch.allclose(eigenvalues_L.real.sort().values,
                              L.diagonal().sort().values, atol=1e-5))

# 对角矩阵
D = torch.diag(torch.tensor([1.0, 5.0, 3.0]))
eigenvalues_D, _ = torch.linalg.eig(D)
print("\n对角矩阵 D 的特征值:", eigenvalues_D.real.tolist(), "= 对角线元素")

### 4.5 特征值的其他性质

- $A$ 和 $A^T$ 有相同的特征值（但特征向量不一定相同）
- $A^k$ 的特征值是 $\lambda_i^k$
- $cA$ 的特征值是 $c\lambda_i$
- $A + cI$ 的特征值是 $\lambda_i + c$

In [ ]:
A = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
eigenvalues_A, _ = torch.linalg.eig(A)
print("A 的特征值:", sorted(eigenvalues_A.real.tolist()))

# A^T 有相同特征值
eigenvalues_AT, _ = torch.linalg.eig(A.T)
print("A^T 的特征值:", sorted(eigenvalues_AT.real.tolist()), "(相同)")

# A^2 的特征值 = λ^2
A2 = A @ A
eigenvalues_A2, _ = torch.linalg.eig(A2)
print("\nA^2 的特征值:", sorted(eigenvalues_A2.real.tolist()))
print("λ(A)^2:", sorted((eigenvalues_A.real**2).tolist()), "(匹配)")

# cA 的特征值 = cλ
c = 3.0
eigenvalues_cA, _ = torch.linalg.eig(c * A)
print("\n3A 的特征值:", sorted(eigenvalues_cA.real.tolist()))
print("3*λ(A):", sorted((3 * eigenvalues_A.real).tolist()), "(匹配)")

# A + cI 的特征值 = λ + c
eigenvalues_ApI, _ = torch.linalg.eig(A + 5 * torch.eye(2))
print("\nA+5I 的特征值:", sorted(eigenvalues_ApI.real.tolist()))
print("λ(A)+5:", sorted((eigenvalues_A.real + 5).tolist()), "(匹配)")

## 5. 特征分解与对角化

### 5.1 特征分解（Eigendecomposition）

如果 $n \times n$ 矩阵 $A$ 有 $n$ 个线性无关的特征向量，将它们作为列组成矩阵 $V$，则：

$$
A = V \Lambda V^{-1}
$$

其中 $\Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_n)$ 是特征值构成的对角矩阵。

这称为 $A$ 的**特征分解**（eigendecomposition），也称**对角化**（diagonalization）。

**可对角化的条件**：$A$ 有 $n$ 个线性无关的特征向量。
- 有 $n$ 个不同特征值的矩阵一定可对角化
- 有重特征值的矩阵可能可对角化，也可能不可对角化（亏损矩阵）

In [ ]:
A = torch.tensor([[4.0, -2.0],
                  [1.0, 1.0]])

eigenvalues, eigenvectors = torch.linalg.eig(A)

# V = 特征向量矩阵（列是特征向量）
V = eigenvectors
# Lambda = 特征值对角矩阵
Lambda = torch.diag(eigenvalues)

print("A =\n", A)
print("\nV (特征向量矩阵) =\n", V)
print("\nLambda (特征值对角矩阵) =\n", Lambda)

# 验证 A = V @ Lambda @ V^{-1}
V_inv = torch.linalg.inv(V)
A_reconstructed = V @ Lambda @ V_inv
print("\nV @ Lambda @ V^{-1} =\n", A_reconstructed.real)
print("\n与 A 相等?", torch.allclose(A_reconstructed.real, A, atol=1e-5))

# 验证 V 的列线性无关（det(V) ≠ 0）
print("\ndet(V) =", torch.det(V).abs().item(), "(≠0 → 特征向量线性无关)")

In [ ]:
# 3x3 矩阵的特征分解
B = torch.tensor([[2.0, 1.0, 0.0],
                  [1.0, 2.0, 1.0],
                  [0.0, 1.0, 2.0]])

eigenvalues_B, eigenvectors_B = torch.linalg.eig(B)
V_B = eigenvectors_B
Lambda_B = torch.diag(eigenvalues_B)

print("B 的特征值:", eigenvalues_B.real.tolist())
print("B 的特征向量（列）:\n", eigenvectors_B.real)

# 重构
B_reconstructed = V_B @ Lambda_B @ torch.linalg.inv(V_B)
print("\n重构 B = VΛV⁻¹:\n", B_reconstructed.real)
print("与原 B 相等?", torch.allclose(B_reconstructed.real, B, atol=1e-5))

# 验证特征向量正交（对称矩阵的特征向量应该正交）
print("\n对称矩阵的特征向量正交性:")
for i in range(3):
    for j in range(i+1, 3):
        dot_ij = torch.dot(eigenvectors_B[:, i].real, eigenvectors_B[:, j].real)
        print(f"  v{i+1} · v{j+1} = {dot_ij.item():.6f} (≈0)")

### 5.2 应用：矩阵幂的快速计算

特征分解最大的应用之一是快速计算矩阵幂：

$$
A^k = (V \Lambda V^{-1})^k = V \Lambda^k V^{-1}
$$

因为 $\Lambda$ 是对角矩阵，$\Lambda^k$ 只需对每个对角线元素取 $k$ 次方：

$$
\Lambda^k = \text{diag}(\lambda_1^k, \lambda_2^k, \ldots, \lambda_n^k)
$$

这将 $O(n^3 \log k)$ 的矩阵快速幂降为 $O(n^3)$（一次分解 + 两次乘法）。

In [ ]:
import time

A = torch.tensor([[1.5, 0.5],
                  [0.5, 1.5]])
k = 50

# 方法1：直接连乘
def matrix_power_direct(A, k):
    result = torch.eye(A.shape[0])
    base = A.clone()
    while k > 0:
        if k % 2 == 1:
            result = result @ base
        base = base @ base
        k //= 2
    return result

Ak_direct = matrix_power_direct(A, k)
print(f"直接计算 A^{k}（快速幂）:\n{Ak_direct}")

# 方法2：特征分解
eigenvalues, eigenvectors = torch.linalg.eig(A)
V = eigenvectors
Lambda_k = torch.diag(eigenvalues ** k)
Ak_eigen = V @ Lambda_k @ torch.linalg.inv(V)
print(f"\n特征分解计算 A^{k}:\n{Ak_eigen.real}")
print("\n两种方法结果相等?", torch.allclose(Ak_direct, Ak_eigen.real, atol=1e-4))

# 验证：Λ^k 就是对角线元素取 k 次方
print(f"\n特征值: {eigenvalues.real.tolist()}")
print(f"特征值^{k}: {(eigenvalues.real ** k).tolist()}")
print("Λ^k 的对角线:", Lambda_k.real.diagonal().tolist())

In [ ]:
# 性能对比：大矩阵高次幂
n = 100
k = 1000
A_big = torch.randn(n, n)
A_big = A_big @ A_big.T + torch.eye(n)  # 对称正定，保证可对角化

# 直接快速幂
start = time.time()
Ak_direct_big = matrix_power_direct(A_big, k)
time_direct = time.time() - start

# 特征分解法
start = time.time()
eigenvalues_big, eigenvectors_big = torch.linalg.eig(A_big)
Lambda_k_big = torch.diag(eigenvalues_big ** k)
Ak_eigen_big = eigenvectors_big @ Lambda_k_big @ torch.linalg.inv(eigenvectors_big)
time_eigen = time.time() - start

print(f"矩阵大小: {n}x{n}, 幂次: {k}")
print(f"直接快速幂耗时: {time_direct*1000:.1f} ms")
print(f"特征分解法耗时: {time_eigen*1000:.1f} ms")
print(f"结果一致? {torch.allclose(Ak_direct_big, Ak_eigen_big.real, atol=1e-2)}")
print("（特征分解法在高次幂时优势明显，且只需一次分解）")

## 6. 对称矩阵的正交对角化

### 6.1 对称矩阵的特殊性质

对称矩阵（$A = A^T$）有三个极其重要的性质：

1. **特征值都是实数**（不会出现复数特征值）
2. **不同特征值对应的特征向量两两正交**
3. **一定可以正交对角化**：存在正交矩阵 $Q$（$Q^T Q = I$，即 $Q^{-1} = Q^T$），使得：

$$
A = Q \Lambda Q^T
$$

这比一般的特征分解 $A = V \Lambda V^{-1}$ 更强——不需要求逆，只需转置。

> 对称矩阵的正交对角化是 PCA、谱聚类、谱归一化等众多算法的数学基础。

In [ ]:
# 对称矩阵的特征值全为实数
A_sym = torch.tensor([[3.0, 1.0, 0.0],
                       [1.0, 2.0, 1.0],
                       [0.0, 1.0, 3.0]])

eigenvalues, eigenvectors = torch.linalg.eig(A_sym)
print("对称矩阵 A =\n", A_sym)
print("\n特征值:", eigenvalues)
print("特征值虚部:", eigenvalues.imag.abs().max().item(), "(≈0 → 全是实数)")
print("特征值（实部）:", eigenvalues.real.tolist())

# 特征向量两两正交
print("\n特征向量正交性验证:")
Q = eigenvectors.real
for i in range(3):
    for j in range(i+1, 3):
        dot_ij = torch.dot(Q[:, i], Q[:, j])
        print(f"  v{i+1} · v{j+1} = {dot_ij.item():.6f} (≈0)")

# Q 是正交矩阵：Q^T Q = I
print("\nQ^T @ Q =\n", Q.T @ Q)
print("≈ I?", torch.allclose(Q.T @ Q, torch.eye(3), atol=1e-5))

# 正交对角化：A = Q @ Λ @ Q^T
Lambda = torch.diag(eigenvalues.real)
A_reconstructed = Q @ Lambda @ Q.T
print("\nQ @ Λ @ Q^T =\n", A_reconstructed)
print("与 A 相等?", torch.allclose(A_reconstructed, A_sym, atol=1e-5))

### 6.2 torch.linalg.eigh：专门用于对称矩阵

`torch.linalg.eigh(A)` 专门用于对称（或 Hermitian）矩阵，相比通用的 `eig`：
- 更快（利用对称性）
- 更稳定（数值精度更高）
- 返回的特征值默认按升序排列
- 特征值一定是实数，直接返回实数 dtype

**注意**：`eigh` 假设输入是对称的，只读取上三角（或下三角）部分。如果输入不是真正对称的，结果可能不正确。

In [ ]:
A_sym = torch.tensor([[3.0, 1.0, 0.0],
                       [1.0, 2.0, 1.0],
                       [0.0, 1.0, 3.0]])

# eigh：返回实数特征值和特征向量
eigenvalues_h, eigenvectors_h = torch.linalg.eigh(A_sym)

print("eigh 返回的特征值（升序排列）:", eigenvalues_h.tolist())
print("eigh 特征值 dtype:", eigenvalues_h.dtype, "(实数！)")
print("\neigh 特征向量（列）:\n", eigenvectors_h)

# 与 eig 对比
eigenvalues_g, eigenvectors_g = torch.linalg.eig(A_sym)
print("\neig 返回的特征值（未排序）:", eigenvalues_g.real.tolist())
print("eig 特征值 dtype:", eigenvalues_g.dtype, "(复数)")

# 验证 eigh 结果
print("\n验证 eigh: A @ v = λv")
for i in range(3):
    lam = eigenvalues_h[i]
    v = eigenvectors_h[:, i]
    Av = A_sym @ v
    print(f"  λ={lam:.4f}, A@v≈λv? {torch.allclose(Av, lam*v, atol=1e-5)}")

# eigh 的正交性
print("\nQ^T @ Q ≈ I?", torch.allclose(eigenvectors_h.T @ eigenvectors_h, torch.eye(3), atol=1e-5))

In [ ]:
# eig vs eigh 性能对比
import time
n = 500
A_big_sym = torch.randn(n, n)
A_big_sym = (A_big_sym + A_big_sym.T) / 2  # 对称化

# eig
start = time.time()
for _ in range(5):
    eigvals_g, _ = torch.linalg.eig(A_big_sym)
time_eig = (time.time() - start) / 5

# eigh
start = time.time()
for _ in range(5):
    eigvals_h, _ = torch.linalg.eigh(A_big_sym)
time_eigh = (time.time() - start) / 5

print(f"矩阵大小: {n}x{n}")
print(f"eig  平均耗时: {time_eig*1000:.1f} ms")
print(f"eigh 平均耗时: {time_eigh*1000:.1f} ms")
print(f"eigh 更快: {time_eig/time_eigh:.1f}x")
print(f"特征值一致? {torch.allclose(eigvals_g.real.sort().values, eigvals_h.sort().values, atol=1e-4)}")

### 6.3 构造对称矩阵并验证正交对角化

任意矩阵 $A$ 都可以通过 $A_{sym} = \frac{A + A^T}{2}$ 构造对称矩阵。

In [ ]:
# 从随机矩阵构造对称矩阵
A_random = torch.randn(4, 4)
A_symmetric = (A_random + A_random.T) / 2

print("随机矩阵 A_random 是否对称?", torch.allclose(A_random, A_random.T, atol=1e-5))
print("对称化后 A_symmetric 是否对称?", torch.allclose(A_symmetric, A_symmetric.T, atol=1e-5))

# eigh 求解
eigenvalues, eigenvectors = torch.linalg.eigh(A_symmetric)
print("\n特征值（升序）:", eigenvalues.tolist())
print("特征值全为实数?", eigenvalues.is_complex() == False)

# 正交对角化验证
Q = eigenvectors
Lambda = torch.diag(eigenvalues)
A_reconstructed = Q @ Lambda @ Q.T
print("\nA = QΛQ^T 重构误差:", (A_reconstructed - A_symmetric).abs().max().item())
print("重构成功?", torch.allclose(A_reconstructed, A_symmetric, atol=1e-5))

# Q 的正交性
print("Q^T Q = I?", torch.allclose(Q.T @ Q, torch.eye(4), atol=1e-5))

## 7. 几何直觉深度可视化

### 7.1 特征值决定拉伸/压缩方向

对于对称矩阵，特征向量方向就是变换的"主轴"，特征值的绝对值决定该方向的拉伸/压缩程度。

用单位圆可视化：单位圆经过线性变换后变成椭圆，椭圆的轴方向就是特征向量方向，轴长度就是特征值的绝对值。

In [ ]:
%matplotlib inline

def visualize_unit_circle_transform(A, title):
    """可视化单位圆经过线性变换后的椭圆，标出特征向量方向"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 生成单位圆
    theta = torch.linspace(0, 2 * 3.14159, 200)
    circle = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    transformed = (A @ circle.T).T
    
    # 左图：单位圆 + 特征向量
    axes[0].plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.7, label='单位圆')
    eigenvalues, eigenvectors = torch.linalg.eig(A)
    for i in range(2):
        if abs(eigenvalues[i].imag.item()) > 1e-6:
            continue
        vec = eigenvectors[:, i].real
        lam = eigenvalues[i].real.item()
        axes[0].arrow(0, 0, vec[0].item(), vec[1].item(),
                      head_width=0.1, color='red', linewidth=2,
                      label=f'v{i+1} (λ={lam:.2f})')
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[0].set_title("单位圆 + 特征向量方向")
    axes[0].set_xlim(-2, 2)
    axes[0].set_ylim(-2, 2)
    
    # 右图：变换后的椭圆
    axes[1].plot(transformed[:, 0], transformed[:, 1], 'r-', alpha=0.7, label='变换后(椭圆)')
    axes[1].fill(transformed[:, 0], transformed[:, 1], 'r', alpha=0.1)
    for i in range(2):
        if abs(eigenvalues[i].imag.item()) > 1e-6:
            continue
        vec = eigenvectors[:, i].real
        lam = eigenvalues[i].real.item()
        transformed_vec = A @ vec
        axes[1].arrow(0, 0, transformed_vec[0].item(), transformed_vec[1].item(),
                      head_width=0.15, color='green', linewidth=2,
                      label=f'A@v{i+1} = {lam:.2f}v{i+1}')
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[1].set_title("变换后: 单位圆 → 椭圆")
    axes[1].set_xlim(-4, 4)
    axes[1].set_ylim(-4, 4)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# 对称矩阵：特征向量正交，是椭圆的轴
A1 = torch.tensor([[3.0, 1.0],
                   [1.0, 0.5]])
visualize_unit_circle_transform(A1, "对称矩阵 [[3,1],[1,0.5]]: 椭圆轴=特征向量方向")

In [ ]:
# 各向同性缩放：特征值相等，圆还是圆
A2 = torch.tensor([[2.0, 0.0],
                   [0.0, 2.0]])
visualize_unit_circle_transform(A2, "各向同性缩放 2I: 特征值相等，圆→更大的圆")

# 一个方向拉伸，一个方向压缩
A3 = torch.tensor([[2.5, 0.0],
                   [0.0, 0.4]])
visualize_unit_circle_transform(A3, "对角 [[2.5,0],[0,0.4]]: x拉伸, y压缩")

### 7.2 旋转矩阵：没有实特征向量

旋转矩阵（除了 0° 和 180°）没有实特征值和实特征向量——因为旋转会改变所有非零向量的方向。

此时特征值是复数 $e^{\pm i\theta} = \cos\theta \pm i\sin\theta$，模为 1。

In [ ]:
import math

# 旋转 45°
theta = math.pi / 4
R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta),  math.cos(theta)]])

eigenvalues, eigenvectors = torch.linalg.eig(R)
print("旋转矩阵 R (45°) =\n", R)
print("\n特征值:", eigenvalues)
print(f"λ1 = {eigenvalues[0].item():.4f} = e^(i·45°)")
print(f"λ2 = {eigenvalues[1].item():.4f} = e^(-i·45°)")
print("|λ1| =", eigenvalues[0].abs().item(), "|λ2| =", eigenvalues[1].abs().item())

# 可视化：旋转后圆还是圆，但所有向量方向都改变了
fig, ax = plt.subplots(figsize=(6, 6))
theta_grid = torch.linspace(0, 2 * math.pi, 200)
circle = torch.stack([torch.cos(theta_grid), torch.sin(theta_grid)], dim=1)
transformed = (R @ circle.T).T

ax.plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.5, label='单位圆')
ax.plot(transformed[:, 0], transformed[:, 1], 'r-', alpha=0.7, label='旋转45°后')

# 画几个向量，都改变了方向
for angle in [0, math.pi/4, math.pi/2]:
    v = torch.tensor([math.cos(angle), math.sin(angle)])
    Rv = R @ v
    ax.arrow(0, 0, v[0], v[1], head_width=0.08, color='blue', alpha=0.6)
    ax.arrow(0, 0, Rv[0], Rv[1], head_width=0.08, color='red', alpha=0.8)

ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("旋转矩阵: 无实特征向量，所有向量方向都改变")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
plt.tight_layout()
plt.show()

print("\n→ 旋转不改变长度（|λ|=1），但改变所有向量的方向，因此没有实特征向量")

### 7.3 特征值的正负与方向反转

- 正特征值：特征向量方向不变
- 负特征值：特征向量方向反转（180°）
- 零特征值：该方向被压缩到零（降维）

In [ ]:
# 负特征值：方向反转
A_neg = torch.tensor([[-2.0, 0.0],
                      [0.0, 1.5]])

eigenvalues, eigenvectors = torch.linalg.eig(A_neg)
print("A =\n", A_neg)
print("特征值:", eigenvalues.real.tolist())
print("λ1 = -2 (负 → 方向反转), λ2 = 1.5 (正 → 方向不变)")

fig, ax = plt.subplots(figsize=(6, 6))
theta_grid = torch.linspace(0, 2 * math.pi, 200)
circle = torch.stack([torch.cos(theta_grid), torch.sin(theta_grid)], dim=1)
transformed = (A_neg @ circle.T).T

ax.plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.4, label='单位圆')
ax.plot(transformed[:, 0], transformed[:, 1], 'r-', alpha=0.7, label='变换后')

# 特征向量
for i in range(2):
    v = eigenvectors[:, i].real
    lam = eigenvalues[i].real.item()
    ax.arrow(0, 0, v[0].item(), v[1].item(), head_width=0.1, color='green', linewidth=2)
    ax.arrow(0, 0, (A_neg @ v)[0].item(), (A_neg @ v)[1].item(),
             head_width=0.1, color='purple', linewidth=2, linestyle='--')

ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("负特征值: x方向反转并拉伸, y方向不变")
ax.set_xlim(-3, 3)
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

# 零特征值：降维
A_zero = torch.tensor([[1.0, 2.0],
                       [2.0, 4.0]])  # det=0，秩=1
eigvals_z, eigvecs_z = torch.linalg.eig(A_zero)
print("\n奇异矩阵 A =\n", A_zero)
print("特征值:", eigvals_z.real.tolist(), "(一个为0 → 降维)")
print("秩:", torch.linalg.matrix_rank(A_zero).item(), "(= 非零特征值个数)")

## 8. 应用入门

### 8.1 斐波那契数列的矩阵幂解法

斐波那契数列：$F_0=0, F_1=1, F_{n+1}=F_n+F_{n-1}$

可以写成矩阵形式：

$$
\begin{pmatrix} F_{n+1} \\ F_n \end{pmatrix} = \begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} F_n \\ F_{n-1} \end{pmatrix}
$$

因此 $\begin{pmatrix} F_n \\ F_{n-1} \end{pmatrix} = A^{n-1} \begin{pmatrix} 1 \\ 0 \end{pmatrix}$，其中 $A = \begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}$。

用特征分解快速计算 $A^n$，可以得到斐波那契数列的通项公式（比内公式）。

In [ ]:
A_fib = torch.tensor([[1.0, 1.0],
                       [1.0, 0.0]])

# 特征分解
eigenvalues, eigenvectors = torch.linalg.eig(A_fib)
print("斐波那契矩阵 A =\n", A_fib)
print("特征值:", eigenvalues.real.tolist())
print(f"λ1 = {(1+5**0.5)/2:.6f} (黄金比例 φ)")
print(f"λ2 = {(1-5**0.5)/2:.6f} (ψ = 1-φ)")

V = eigenvectors
V_inv = torch.linalg.inv(V)

def fib_eigen(n):
    """用特征分解计算第 n 个斐波那契数"""
    if n == 0:
        return 0
    if n == 1:
        return 1
    Lambda_n = torch.diag(eigenvalues ** (n - 1))
    A_n = V @ Lambda_n @ V_inv
    result = A_n.real @ torch.tensor([1.0, 0.0])
    return round(result[0].item())

def fib_direct(n):
    """直接递推计算"""
    if n == 0: return 0
    if n == 1: return 1
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

# 验证前 20 项
print("\n斐波那契数列验证:")
print(f"{'n':>4} {'特征分解法':>12} {'直接递推':>12} {'匹配':>6}")
for n in [0, 1, 2, 3, 5, 10, 15, 20, 30, 50]:
    f_eigen = fib_eigen(n)
    f_direct = fib_direct(n)
    print(f"{n:>4} {f_eigen:>12} {f_direct:>12} {'✓' if f_eigen == f_direct else '✗':>6}")

# 比内公式验证
def fib_binet(n):
    phi = (1 + 5**0.5) / 2
    psi = (1 - 5**0.5) / 2
    return round((phi**n - psi**n) / 5**0.5)

print("\n比内公式验证 (n=50):", fib_binet(50), "==", fib_direct(50), "?", fib_binet(50) == fib_direct(50))
print("→ 特征分解的极限形式就是比内公式")

### 8.2 PCA 主成分分析入门

PCA（Principal Component Analysis）是最常用的降维方法，其数学基础就是对称矩阵的正交对角化。

**PCA 步骤**：
1. 数据中心化（每列减去均值）
2. 计算协方差矩阵 $\Sigma = \frac{1}{n-1} X^T X$（对称矩阵）
3. 对 $\Sigma$ 做特征分解：$\Sigma = Q \Lambda Q^T$
4. 特征值从大到小排列，对应的特征向量就是主成分方向
5. 将数据投影到前 k 个主成分上实现降维

**特征值的意义**：第 i 个特征值 = 第 i 个主成分方向上的方差。特征值越大，该方向包含的信息越多。

In [ ]:
# PCA 完整实例：2D 数据降维到 1D
torch.manual_seed(42)

# 生成有相关性的 2D 数据
n_samples = 200
x1 = torch.randn(n_samples)
x2 = 2.0 * x1 + 0.5 * torch.randn(n_samples)  # x2 与 x1 强相关
X = torch.stack([x1, x2], dim=1)  # [200, 2]

print("数据形状:", X.shape)
print("数据均值:", X.mean(dim=0).tolist())

# Step 1: 中心化
X_centered = X - X.mean(dim=0)
print("中心化后均值:", X_centered.mean(dim=0).tolist(), "(≈0)")

# Step 2: 协方差矩阵
cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)
print("\n协方差矩阵 Σ =\n", cov_matrix)
print("Σ 是对称矩阵?", torch.allclose(cov_matrix, cov_matrix.T, atol=1e-5))

# Step 3: 特征分解（用 eigh，因为协方差矩阵对称）
eigenvalues, eigenvectors = torch.linalg.eigh(cov_matrix)
# eigh 返回升序，我们需要降序
eigenvalues_desc = eigenvalues.flip(0)
eigenvectors_desc = eigenvectors.flip(1)

print("\n特征值（降序）:", eigenvalues_desc.tolist())
print("特征向量（列，降序）:\n", eigenvectors_desc)
print("\n方差解释比例:", (eigenvalues_desc / eigenvalues_desc.sum()).tolist())
print(f"第一主成分解释了 {eigenvalues_desc[0]/eigenvalues_desc.sum()*100:.1f}% 的方差")

In [ ]:
# Step 4 & 5: 可视化 + 投影降维
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左图：原始数据 + 主成分方向
axes[0].scatter(X_centered[:, 0].numpy(), X_centered[:, 1].numpy(),
                alpha=0.5, s=20, label='中心化数据')
for i in range(2):
    vec = eigenvectors_desc[:, i]
    lam = eigenvalues_desc[i].item()
    # 按特征值缩放箭头长度
    scale = (lam ** 0.5) * 2
    axes[0].arrow(0, 0, vec[0].item() * scale, vec[1].item() * scale,
                  head_width=0.15, color='red', linewidth=2,
                  label=f'PC{i+1} (λ={lam:.2f}, {lam/eigenvalues_desc.sum()*100:.1f}%)')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[0].set_title("原始数据 + 主成分方向")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

# 右图：投影到第一主成分（降维到 1D）
pc1 = eigenvectors_desc[:, 0:1]  # [2, 1]
X_projected = X_centered @ pc1  # [200, 1]
print("投影后形状:", X_projected.shape, "(从 2D 降到 1D)")

# 重构：从 1D 投影重构回 2D
X_reconstructed = X_projected @ pc1.T  # [200, 2]

axes[1].scatter(X_centered[:, 0].numpy(), X_centered[:, 1].numpy(),
                alpha=0.2, s=20, label='原始')
axes[1].scatter(X_reconstructed[:, 0].numpy(), X_reconstructed[:, 1].numpy(),
                alpha=0.5, s=20, color='red', label='PC1 重构')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_title(f"降维到 PC1 (保留 {eigenvalues_desc[0]/eigenvalues_desc.sum()*100:.1f}% 方差)")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")

plt.tight_layout()
plt.show()

# 计算重构误差
reconstruction_error = ((X_centered - X_reconstructed) ** 2).sum() / n_samples
print(f"\n平均重构误差: {reconstruction_error.item():.4f}")
print(f"= 第二主成分的方差 (λ2): {eigenvalues_desc[1].item():.4f}")
print("→ 丢弃的方差 = 被丢弃主成分的特征值之和")

In [ ]:
# 3D 数据降维到 2D 的 PCA 实例
torch.manual_seed(123)
n = 300

# 生成 3D 数据，主要在一个 2D 平面上变化
z1 = torch.randn(n)
z2 = torch.randn(n)
x = z1 + 0.1 * torch.randn(n)
y = 0.5 * z1 + z2 + 0.1 * torch.randn(n)
z = 0.3 * z1 + 0.7 * z2 + 0.1 * torch.randn(n)
X3d = torch.stack([x, y, z], dim=1)

# PCA
X3d_centered = X3d - X3d.mean(dim=0)
cov3d = (X3d_centered.T @ X3d_centered) / (n - 1)
eigvals3d, eigvecs3d = torch.linalg.eigh(cov3d)
eigvals3d = eigvals3d.flip(0)
eigvecs3d = eigvecs3d.flip(1)

print("3D 数据协方差矩阵的特征值:", eigvals3d.tolist())
print("方差解释比例:", (eigvals3d / eigvals3d.sum()).tolist())
print(f"前两个主成分累计解释 {eigvals3d[:2].sum()/eigvals3d.sum()*100:.1f}% 方差")

# 投影到 2D
pc2 = eigvecs3d[:, :2]
X3d_projected = X3d_centered @ pc2
print(f"\n投影后形状: {X3d.shape} → {X3d_projected.shape}")

# 可视化
fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X3d_centered[:, 0], X3d_centered[:, 1], X3d_centered[:, 2],
            alpha=0.4, s=15)
ax1.set_title("原始 3D 数据")
ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("z")

ax2 = fig.add_subplot(122)
ax2.scatter(X3d_projected[:, 0], X3d_projected[:, 1],
            alpha=0.5, s=15, c='red')
ax2.set_title(f"PCA 降维到 2D ({eigvals3d[:2].sum()/eigvals3d.sum()*100:.1f}% 方差)")
ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 课后练习

### 基础题

1. 对矩阵 $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$，手算特征多项式 $\det(A-\lambda I)$，求出特征值，再用 `torch.linalg.eig` 验证。

2. 对矩阵 $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$，用 `torch.linalg.eig` 求特征值和特征向量，逐对验证 $Av = \lambda v$。

3. 创建一个 $5 \times 5$ 的随机矩阵，验证：特征值之和 = 迹，特征值之积 = 行列式。

### 验证题

4. 对对称矩阵 $A = \begin{pmatrix} 3 & 1 & 0 \\ 1 & 2 & 1 \\ 0 & 1 & 3 \end{pmatrix}$，分别用 `eig` 和 `eigh` 求特征值，对比结果（排序、dtype、数值精度）。

5. 验证对称矩阵的正交对角化：对上述矩阵构造 $Q$ 和 $\Lambda$，验证 $A = Q\Lambda Q^T$ 且 $Q^T Q = I$。

6. 验证矩阵幂公式：取任意可对角化矩阵 $A$，用特征分解计算 $A^{10}$，与直接连乘结果对比。

### 几何题

7. 对矩阵 $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$，画出单位圆经过变换后的椭圆，标出特征向量方向，验证椭圆的轴就是特征向量方向。

8. 对旋转矩阵 $R(30°)$，验证其特征值是复数且模为 1，解释为什么旋转矩阵没有实特征向量。

### 综合题

9. 用特征分解法计算第 100 个斐波那契数，与直接递推法对比结果和耗时。

10. 生成一个 4D 数据集（1000 个样本，各维度之间有相关性），用 PCA 降维到 2D，画出碎石图（scree plot，特征值柱状图），确定保留多少主成分合适，并可视化降维结果。

### autograd 衔接题

11. 设置 `A = torch.randn(3, 3, requires_grad=True)`，令 `A_sym = (A + A.T) / 2`，计算 `loss = torch.linalg.eigvalsh(A_sym).sum()`（即对称化矩阵的迹），调用 `backward()`，验证 `A.grad` 是否等于 $I$（因为迹对矩阵的梯度是单位矩阵）。

12. 对一个 2×2 对称矩阵 $A$ 设置 `requires_grad=True`，计算 `loss = (torch.linalg.eigvalsh(A)[0])`（最小特征值），调用 `backward()`，观察 `A.grad`，尝试推导最小特征值对矩阵元素的梯度公式（提示：$\frac{\partial \lambda}{\partial A} = vv^T$，其中 $v$ 是对应特征向量）。